# 03 · FlashAttention

> 上一节通过 Online Softmax 推导了在不保存完整 $S/A$ 矩阵的前提下完成注意力计算的算法。本节从硬件视角出发，定量分析 FlashAttention 的 HBM 访问量，通过简化的 SRAM 容量模型理解 tile size 的选择依据，并通过 PyTorch 内置的 SDPA 接口实测加速效果。

**目录**
1. [IO 复杂度分析](#1-IO-复杂度分析)
2. [Tiling 算法完整伪代码](#2-Tiling-算法完整伪代码)
3. [`F.scaled_dot_product_attention`](#3-F.scaled_dot_product_attention)
4. [GPU 基准测试](#4-GPU-基准测试)
5. [反向传播：重计算策略](#5-反向传播：重计算策略)
6. [FlashAttention v2 改进](#6-FlashAttention-v2-改进)
7. [小结](#7-小结)

---

In [ ]:
import torch
import torch.nn.functional as F
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import json
import os

matplotlib.rcParams["font.family"] = ["DejaVu Sans", "sans-serif"]
os.makedirs("../results", exist_ok=True)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA : {torch.cuda.is_available()}")
device = "cuda"if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {props.name}")
    print(f"总显存: {props.total_memory / 1e9:.1f} GB")
    print(f"SM 数量: {props.multi_processor_count}")

PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : NVIDIA GeForce RTX 4060 Laptop GPU
  总显存  : 8.6 GB
  SM 数量 : 24


## 1 IO 复杂度分析

标准注意力的前向传播需要把 $N \times N$ 的中间矩阵 $S$ 和 $A$ 写入 HBM 再读回：

```
步骤 1: 读 Q(Nd), K(Nd) → 计算 S = QKᵀ/√d → 写 S(N²)    读 2Nd + 写 N²
步骤 2: 读 S(N²)         → 计算 softmax(S)  → 写 A(N²)    读 N²  + 写 N²
步骤 3: 读 A(N²), V(Nd)  → 计算 AV          → 写 O(Nd)    读 N²+Nd + 写 Nd
────────────────────────────────────────────────────────────────
合计：Θ(Nd + N²)  （N 较大时 N² 项主导）
```

FlashAttention 的核心想法：把 $Q$、$K$、$V$ 的小块装进 SRAM，在 SRAM 内完成 $S_{ij}$、softmax、$\tilde{P}_{ij}V_j$ 的全部计算，$S$ 和 $A$ 始终不离开 SRAM，从而避免了那几次 $O(N^2)$ 的 HBM 读写。

### 1.1 FlashAttention 的 HBM 访问量推导

设 SRAM 容量为 $M$ 个元素。Tiling 时 SRAM 中需要同时容纳 $Q_i$（$B_r \times d$）、$K_j$（$B_c \times d$）、$V_j$（$B_c \times d$）以及局部 logits $S_{ij}$（$B_r \times B_c$），约束为：
$$
B_r d + 2 B_c d + B_r B_c \le M
$$

忽略 $B_r B_c$ 项（相对较小），令 $B_r \approx B_c \approx B$，化简得 $B \approx M/(3d)$。FlashAttention 论文取 $B_c \approx M/(4d)$（保留裕量），下面用 $B_c \approx M/(4d)$ 推导复杂度。

**HBM 访问量（Algorithm 1：外层 $K/V$，内层 $Q$）**

设块数 $T_c = \lceil N/B_c \rceil$，$T_r = \lceil N/B_r \rceil$：

- $K$、$V$ 各被外层读一次：$T_c \cdot 2B_c d = 2Nd$
- $Q$ 每次外层迭代被内层读完整一遍：$T_c \cdot Nd$
- $O_i$、$l_i$、$m_i$ 每次内层迭代各读写一次，量级同 $Q$：$T_c \cdot O(Nd)$

主导项为 $Q$/$O$ 的读写：
$$
\text{HBM 访问量} = O(T_c \cdot Nd) = O\!\left(\frac{N}{B_c} \cdot Nd\right) = O\!\left(\frac{N^2 d}{B_c}\right)
$$

代入 $B_c \approx M/(4d)$：
$$
\boxed{\text{FlashAttention HBM 访问量} = \Theta\!\left(\frac{N^2 d^2}{M}\right)}
$$

**相比标准注意力的节省倍数**

标准注意力主导项为 $\Theta(N^2)$，因此：
$$
\frac{\Theta(N^2)}{\Theta(N^2 d^2 / M)} = \frac{M}{d^2}
$$

以 A100 FP16（SRAM $\approx$ 96,000 元素）、$d = 64$ 为例：$M/d^2 \approx 96000/4096 \approx 23\times$。
注意这是 **HBM 访问量的理论节省比**，实测延迟加速受算术强度、kernel 启动开销等多重因素影响，通常在 2–4× 范围内。

### 1.2 简化 SRAM 容量分析与 tile size

上一节得到的 $B_c \approx M/(4d)$ 可以作为理解 SRAM 容量如何限制 tile 大小 的一个简化估算。其中 (M) 表示片上 SRAM 可容纳的元素数量，(d) 为 head dimension。它说明：在 (d) 固定时，更大的片上存储空间允许使用更大的 tile。

但在实际 FlashAttention 实现中，tile size 并不是仅由 SRAM 容量决定的，还需要同时考虑 GPU 的计算与存储结构

因此，$B_c \approx M/(4d)$ 更适合看作一个容量层面的理论估算，而不是实际 kernel 中 tile size 的直接计算公式。

实际的 FlashAttention kernel 会针对具体 GPU 架构、数据类型和 head dimension 进行调优，常见的 tile 维度通常为 (64)、(128) 等适合 GPU 硬件执行的大小。

In [ ]:
# ── IO 复杂度对比可视化 ──────────────────────────────────────────────────────
seq_lens = [128, 256, 512, 1024, 2048, 4096, 8192]
d_k = 64

# A100 FP16: SRAM ~192 KB / 2 bytes per FP16 = 98304 elements
M_fp16 = 192 * 1024 // 2      # ~98304 FP16 elements
# 按字节计算
bytes_fp16 = 2
bytes_fp32 = 4

# 标准注意力（FP32）：Θ(Nd + N²) 字节
std_hbm_bytes   = [(N * d_k + N**2) * bytes_fp32 for N in seq_lens]
# FlashAttention（FP16）：Θ(N²d²/M) 字节
flash_hbm_bytes = [(N**2 * d_k**2 / M_fp16) * bytes_fp16 for N in seq_lens]

std_hbm_mb   = [b / 1e6 for b in std_hbm_bytes]
flash_hbm_mb = [b / 1e6 for b in flash_hbm_bytes]
speedup      = [s / f for s, f in zip(std_hbm_bytes, flash_hbm_bytes)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：HBM 访问量（对数坐标）
axes[0].semilogy(seq_lens, std_hbm_mb, marker="s", color="tomato",
                 linewidth=2, label=r"Standard $\Theta(Nd + N^2)$  [FP32]")
axes[0].semilogy(seq_lens, flash_hbm_mb, marker="o", color="steelblue",
                 linewidth=2, label=r"FlashAttention $\Theta(N^2d^2/M)$  [FP16]")
axes[0].set_xlabel("Sequence Length  $N$", fontsize=11)
axes[0].set_ylabel("HBM Access  (MB, log scale)", fontsize=11)
axes[0].set_title(f"IO Complexity Comparison  ($d$={d_k})")
axes[0].set_xticks(seq_lens)
axes[0].set_xticklabels([str(n) for n in seq_lens], rotation=30)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3, which="both")

# 右：理论加速倍数 Standard/Flash
axes[1].plot(seq_lens, speedup, marker="D", color="seagreen", linewidth=2)
axes[1].set_xlabel("Sequence Length  $N$", fontsize=11)
axes[1].set_ylabel("HBM Access Reduction Factor", fontsize=11)
axes[1].set_title("Theoretical HBM Access Reduction  (Standard / Flash)")
axes[1].set_xticks(seq_lens)
axes[1].set_xticklabels([str(n) for n in seq_lens], rotation=30)
for x, y in zip(seq_lens, speedup):
    axes[1].annotate(f"{y:.0f}×", xy=(x, y), xytext=(0, 6),
                     textcoords="offset points", ha="center", fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle("FlashAttention IO Complexity Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/03_io_complexity.png", dpi=120, bbox_inches="tight")
plt.show()
print("图像已保存至 results/03_io_complexity.png")

# 打印表格
print(f"\n{'N':>6}  {'Standard(MB)':>14}  {'Flash(MB)':>12}  {'节省倍数':>10}")
print("-"* 50)
for n, s, f, x in zip(seq_lens, std_hbm_mb, flash_hbm_mb, speedup):
    print(f"{n:>6}  {s:>14.2f}  {f:>12.4f}  {x:>10.0f}×")

## 2 Tiling 算法完整伪代码

下面以**逐行注释**的方式对照论文原文（Algorithm 1），厘清每一步的 IO 行为。上一节实现的 `tiled_online_attention` 与此伪代码逐行对应。

```
===== FlashAttention Algorithm 1（前向，Dao et al. 2022）=====

输入：Q, K, V ∈ ℝ^{N×d}（存于 HBM）
参数：块大小 Br, Bc

第 1 行：将 K, V 分成 Tc = ⌈N/Bc⌉ 块 K₁…K_Tc, V₁…V_Tc
         将 Q  分成 Tr = ⌈N/Br⌉ 块 Q₁…Q_Tr
         将 O, ℓ, m 按同样方式分块（O 初始化为 0，ℓ 为 0，m 为 -∞）
         -- IO：仅初始化 HBM 上的 O, ℓ, m，无额外读写

第 2 行：for j = 1 to Tc:            # 外循环：K/V 列块
             从 HBM 加载 Kⱼ, Vⱼ 到 SRAM
             -- IO：读 2·Bc·d 个元素（一次，后续内循环复用）

第 3 行：    for i = 1 to Tr:        # 内循环：Q 行块
                 从 HBM 加载 Qᵢ, Oᵢ, ℓᵢ, mᵢ 到 SRAM
                 -- IO：读 Br·d + Br·d + Br + Br ≈ 2·Br·d 个元素

第 4 行：        Sᵢⱼ = Qᵢ · Kⱼᵀ / √d            # (Br, Bc)  全在 SRAM 内完成

第 5 行：        m̃ᵢⱼ = rowmax(Sᵢⱼ)              # (Br,)     SRAM
                 P̃ᵢⱼ = exp(Sᵢⱼ − m̃ᵢⱼ)           # (Br, Bc) SRAM
                 ℓ̃ᵢⱼ = rowsum(P̃ᵢⱼ)              # (Br,)     SRAM
                 -- 关键：S 和 A（即 P̃）从不写回 HBM

第 6 行：        mᵢ_new = max(mᵢ, m̃ᵢⱼ)           # Online Softmax 合并
                 ℓᵢ_new = exp(mᵢ − mᵢ_new)·ℓᵢ
                         + exp(m̃ᵢⱼ − mᵢ_new)·ℓ̃ᵢⱼ

第 7 行：        Oᵢ_new = diag(ℓᵢ_new)⁻¹ · (
                              diag(ℓᵢ)·exp(mᵢ − mᵢ_new)·Oᵢ
                            + exp(m̃ᵢⱼ − mᵢ_new) · P̃ᵢⱼ · Vⱼ
                          )

第 8 行：        将 Oᵢ, ℓᵢ, mᵢ 写回 HBM
                 -- IO：写 ≈ 2·Br·d 个元素

输出：O（HBM 中的完整输出矩阵）

────────── 总 HBM 访问量 ──────────
读 K, V：  Tc × 2·Bc·d ≈ 2Nd
读写 Q,O：  Tc × O(Nd) ≈ N²d/Bc ≈ 4N²d²/M
合计：Θ(N²d²/M)    （与 §1 推导一致）
```

### 关键设计决策

| 设计点 | 原因 |
|--------|------|
| 外循环遍历 $K/V$ 块 | $K_j, V_j$ 在内循环中被所有 $Q_i$ 块复用，只需从 HBM 读一次 |
| $O_i$ 每内循环步均写回 HBM | $Q$ 块多、SRAM 不够同时保留所有 $O_i$；但因块小，写回代价可控 |
| 不写 $S_{ij}$ 或 $A_{ij}$ | 这两个 $B_r \times B_c$ 中间矩阵消费完即丢，每块仅在 SRAM 存活一个 step |
| 维护 $(m, \ell, O)$ 而非完整 $A$ | $O(N)$ 存储 vs $O(N^2)$，这是显存节省的根本来源 |

In [ ]:
# ── Algorithm 1 PyTorch 实现（带详细注释，与上节伪代码逐行对应）────────────────
def flash_attention_v1_forward(
    Q: torch.Tensor, # (B, H, N, d)
    K: torch.Tensor, # (B, H, N, d)
    V: torch.Tensor, # (B, H, N, d)
    block_r: int = 64, # Br：Q 块大小
    block_c: int = 64, # Bc：K/V 块大小
    causal: bool = False,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    FlashAttention v1 前向伪代码的 PyTorch 复现（Algorithm 1）。

    除输出 O 外，还返回 (l, m) 统计量——反向传播重计算所需。
    这与真实 CUDA kernel 保存 log-sum-exp 的做法对应。

    注意：此实现用于算法演示，PyTorch autograd 仍会保留中间变量，
    不等于真正的 O(N) 显存（需要自定义 CUDA kernel 才能实现）。
    """
    B, H, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    Tr = math.ceil(N / block_r)
    Tc = math.ceil(N / block_c)

    # Algorithm 1, Line 1：初始化 O, ℓ, m（HBM 上）
    O = torch.zeros_like(Q)                                              # (B,H,N,d)
    l = torch.zeros(B, H, N, device=Q.device, dtype=Q.dtype)            # (B,H,N)
    m = torch.full((B, H, N), float("-inf"), device=Q.device, dtype=Q.dtype)  # (B,H,N)

    # Algorithm 1, Line 3：外循环 j（K/V 列块）
    for j in range(Tc):
        j0, j1 = j * block_c, min((j + 1) * block_c, N)
        # "从 HBM 加载 Kⱼ, Vⱼ 到 SRAM"
        Kj = K[:,:, j0:j1,:]    # (B,H,Bc',d)
        Vj = V[:,:, j0:j1,:]    # (B,H,Bc',d)

        # Algorithm 1, Line 4：内循环 i（Q 行块）
        for i in range(Tr):
            i0, i1 = i * block_r, min((i + 1) * block_r, N)
            # "从 HBM 加载 Qᵢ, Oᵢ, ℓᵢ, mᵢ 到 SRAM"
            Qi  = Q[:,:, i0:i1,:]   # (B,H,Br',d)
            Oi  = O[:,:, i0:i1,:]   # (B,H,Br',d)
            li  = l[:,:, i0:i1]      # (B,H,Br')
            mi  = m[:,:, i0:i1]      # (B,H,Br')

            # Algorithm 1, Line 6：计算局部 logits Sᵢⱼ（全在 SRAM 内）
            Sij = torch.matmul(Qi, Kj.transpose(-2, -1)) * scale  # (B,H,Br',Bc')

            # 因果掩码（遮蔽 k_pos > q_pos 的位置）
            if causal:
                q_pos = torch.arange(i0, i1, device=Q.device).view(1, 1, -1, 1)
                k_pos = torch.arange(j0, j1, device=Q.device).view(1, 1, 1, -1)
                Sij = Sij.masked_fill(k_pos > q_pos, float("-inf"))

            # Algorithm 1, Line 7：局部统计量（不写 S/P 到 HBM）
            m_tilde = Sij.amax(dim=-1)                              # (B,H,Br')
            P_tilde = torch.exp(Sij - m_tilde.unsqueeze(-1))       # (B,H,Br',Bc')
            l_tilde = P_tilde.sum(dim=-1)                           # (B,H,Br')

            # Algorithm 1, Line 10：合并全局 (m, ℓ)（Online Softmax 递推规则）
            mi_new = torch.maximum(mi, m_tilde)
            li_new = (torch.exp(mi - mi_new) * li
                      + torch.exp(m_tilde - mi_new) * l_tilde)

            # Algorithm 1, Line 12：累积输出 Oᵢ（在 SRAM 内完成）
            alpha = (torch.exp(mi - mi_new) * li / li_new).unsqueeze(-1)    # (B,H,Br',1)
            beta  = (torch.exp(m_tilde - mi_new) / li_new).unsqueeze(-1)    # (B,H,Br',1)
            Oi_new = alpha * Oi + beta * torch.matmul(P_tilde, Vj)

            # "将 Oᵢ, ℓᵢ, mᵢ 写回 HBM"
            O[:,:, i0:i1,:] = Oi_new
            l[:,:, i0:i1]    = li_new
            m[:,:, i0:i1]    = mi_new

    return O, l, m  # 返回 (l, m) 供反向传播使用


# ── 正确性验证 ────────────────────────────────────────────────────────────────
torch.manual_seed(42)
B, H, N_val, d_val = 2, 4, 128, 64

Qv = torch.randn(B, H, N_val, d_val, device=device)
Kv = torch.randn(B, H, N_val, d_val, device=device)
Vv = torch.randn(B, H, N_val, d_val, device=device)

# 参考：PyTorch math backend（精确等价标准注意力）
if device == "cuda":
    with torch.backends.cuda.sdp_kernel(
        enable_flash=False, enable_math=True, enable_mem_efficient=False
    ):
        ref = F.scaled_dot_product_attention(Qv, Kv, Vv)
else:
    ref = F.scaled_dot_product_attention(Qv, Kv, Vv)

print("=== Algorithm 1 正确性验证（非因果）===")
print(f"{'块大小 Br=Bc':>14}  {'最大绝对误差':>14}  通过")
print("-"* 40)
for bs in [8, 16, 32, 64, 128]:
    out, _, _ = flash_attention_v1_forward(Qv, Kv, Vv, block_r=bs, block_c=bs)
    err = (out - ref).abs().max().item()
    print(f"{bs:>14}  {err:>14.2e}  {'✓' if err < 1e-4 else '✗'}")

print("\n=== Algorithm 1 正确性验证（因果）===")
if device == "cuda":
    with torch.backends.cuda.sdp_kernel(
        enable_flash=False, enable_math=True, enable_mem_efficient=False
    ):
        ref_c = F.scaled_dot_product_attention(Qv, Kv, Vv, is_causal=True)
else:
    ref_c = F.scaled_dot_product_attention(Qv, Kv, Vv, is_causal=True)

for bs in [8, 32, 64]:
    out_c, _, _ = flash_attention_v1_forward(Qv, Kv, Vv, block_r=bs, block_c=bs, causal=True)
    err_c = (out_c - ref_c).abs().max().item()
    print(f"{bs:>14}  {err_c:>14.2e}  {'✓' if err_c < 1e-4 else '✗'}")

## 3 `F.scaled_dot_product_attention`

PyTorch 2.0 起，`torch.nn.functional.scaled_dot_product_attention`（SDPA）内置了多个 backend。默认模式下 PyTorch 根据当前输入的数据类型、形状、设备及掩码等条件**自动选择**可用的 backend，并不保证固定的优先级顺序。

### 3.1 可用 Backend

| Backend | 简述 |
|---------|------|
| `FLASH_ATTENTION` | 基于 FlashAttention/cuDNN 的 CUDA kernel；需要 CUDA 设备、fp16/bf16、head_dim ≤ 128，且不使用任意自定义掩码 |
| `EFFICIENT_ATTENTION` | 基于 xFormers memory-efficient attention；覆盖更宽泛的精度与形状，但不及 Flash 快 |
| `MATH` | 等价于手写 `softmax(QKᵀ/√d) @ V`；任意设备/精度均可用，无显存优化 |

由于 PyTorch 的 backend 选择逻辑随版本变化，**不应假设**在 GPU + fp16 条件下一定会调用 FlashAttention kernel。若需确认或强制指定 backend，应使用 `sdp_kernel` 上下文管理器。

### 3.2 `sdp_kernel` 上下文管理器

`torch.backends.cuda.sdp_kernel(enable_flash, enable_math, enable_mem_efficient)` 允许显式控制哪些 backend 参与候选：

```python
# 强制只用 FlashAttention（不满足条件时抛出异常，而非 fallback）
with torch.backends.cuda.sdp_kernel(
    enable_flash=True, enable_math=False, enable_mem_efficient=False
):
    out = F.scaled_dot_product_attention(Q, K, V)

# 强制 math backend（用于精度对比基准）
with torch.backends.cuda.sdp_kernel(
    enable_flash=False, enable_math=True, enable_mem_efficient=False
):
    out_ref = F.scaled_dot_product_attention(Q, K, V)
```

常见用途：
- **精度对比**：强制 math backend 作为参考基准，验证其他 backend 的数值等价性
- **性能对比**：逐个启用各 backend 测量延迟和显存
- **调试**：在不满足 Flash 条件时，禁用 fallback 以暴露问题

In [ ]:
# ── Backend 可用性检查与数值等价性验证 ────────────────────────────────────────
torch.manual_seed(0)
B_b, H_b, N_b, d_b = 2, 8, 512, 64

# fp16：触发 Flash Attention backend
Qb16 = torch.randn(B_b, H_b, N_b, d_b, device=device, dtype=torch.float16)
Kb16 = torch.randn(B_b, H_b, N_b, d_b, device=device, dtype=torch.float16)
Vb16 = torch.randn(B_b, H_b, N_b, d_b, device=device, dtype=torch.float16)

# fp32：仅 math / mem-efficient backend
Qb32 = Qb16.float()
Kb32 = Kb16.float()
Vb32 = Vb16.float()

print("=== SDPA Backend 可用性 ===")
if device == "cuda":
    for attr, name in [
        ("flash_sdp_enabled", "Flash Attention"),
        ("mem_efficient_sdp_enabled", "Mem-Efficient Attention"),
        ("math_sdp_enabled", "Math (fallback)"),
    ]:
        try:
            val = getattr(torch.backends.cuda, attr)()
        except AttributeError:
            val = "N/A（PyTorch 版本不支持）"
        print(f"{name:30s}: {val}")
else:
    print("CPU 模式：仅 math backend 可用")

print()

# 参考：math backend（精确）
if device == "cuda":
    with torch.backends.cuda.sdp_kernel(
        enable_flash=False, enable_math=True, enable_mem_efficient=False
    ):
        ref_math = F.scaled_dot_product_attention(Qb32, Kb32, Vb32)
else:
    ref_math = F.scaled_dot_product_attention(Qb32, Kb32, Vb32)

# 各 backend 与 math 的误差对比
backends_to_test = []
if device == "cuda":
    backends_to_test = [
        ("default (fp16)", lambda: F.scaled_dot_product_attention(Qb16, Kb16, Vb16), True),
        ("default (fp32)", lambda: F.scaled_dot_product_attention(Qb32, Kb32, Vb32), False),
        ("math (fp32)", lambda: _sdpa_math(Qb32, Kb32, Vb32), False),
        ("mem-eff (fp32)", lambda: _sdpa_mem_eff(Qb32, Kb32, Vb32), False),
    ]
    def _sdpa_math(q, k, v):
        with torch.backends.cuda.sdp_kernel(
            enable_flash=False, enable_math=True, enable_mem_efficient=False
        ):
            return F.scaled_dot_product_attention(q, k, v)

    def _sdpa_mem_eff(q, k, v):
        with torch.backends.cuda.sdp_kernel(
            enable_flash=False, enable_math=False, enable_mem_efficient=True
        ):
            return F.scaled_dot_product_attention(q, k, v)

print("=== 各 Backend 与 math 基准的误差对比 ===")
print(f"{'Backend':>22}  {'最大绝对误差':>14}")
print("-"* 40)
for name, fn, is_fp16 in backends_to_test:
    try:
        out = fn()
        if is_fp16:
            out = out.float()
        err = (out - ref_math).abs().max().item()
        print(f"{name:>22}  {err:>14.2e}")
    except Exception as e:
        print(f"{name:>22}  ERROR: {e}")

if device != "cuda":
    out_cpu = F.scaled_dot_product_attention(Qb32, Kb32, Vb32)
    err_cpu = (out_cpu - ref_math).abs().max().item()
    print(f"{'CPU math (fp32)':>22}  {err_cpu:>14.2e}")

print("\n结论：不同 backend 仅有浮点运算顺序差异，误差均在 FP16 精度范围内。")
print("FlashAttention 是精确（exact）算法，而非近似算法。")

In [ ]:
# ── 因果注意力与自定义掩码演示 ────────────────────────────────────────────────
torch.manual_seed(1)
B_c, H_c, N_c, d_c = 1, 4, 64, 32
Qc = torch.randn(B_c, H_c, N_c, d_c, device=device)
Kc = torch.randn(B_c, H_c, N_c, d_c, device=device)
Vc = torch.randn(B_c, H_c, N_c, d_c, device=device)

# 方式 1：is_causal=True（推荐，Flash Attention kernel 原生支持）
out_causal_flag = F.scaled_dot_product_attention(Qc, Kc, Vc, is_causal=True)

# 方式 2：显式传入下三角掩码（-inf 掩蔽上三角）
attn_mask = torch.zeros(N_c, N_c, device=device)
attn_mask = attn_mask.masked_fill(
    torch.triu(torch.ones(N_c, N_c, dtype=torch.bool, device=device), diagonal=1),
    float("-inf")
)
out_causal_mask = F.scaled_dot_product_attention(Qc, Kc, Vc, attn_mask=attn_mask)

err_mask = (out_causal_flag - out_causal_mask).abs().max().item()
print(f"is_causal=True vs 显式掩码的误差: {err_mask:.2e}")
assert err_mask < 1e-4, "两种因果掩码方式结果不一致！"
print("✓ 两种因果掩码方式数学等价")
print()
print("注意：传入 attn_mask 时，Flash Attention kernel 可能不被触发")
print("（cuDNN Flash 不支持任意形状掩码）。is_causal=True 是首选写法。")

## 4 GPU 基准测试

对比三种实现在不同序列长度下的**延迟**和**峰值显存**：

| 实现 | 说明 |
|------|------|
| `standard (math)` | `F.sdpa` 强制 math backend，等价于标准朴素注意力，作为基准 |
| `flash_sim (py)` | 本节 `flash_attention_v1_forward` 的 Python 级 Tiling 模拟 |
| `PyTorch SDPA (auto)` | `F.sdpa` 默认模式，PyTorch 自动选择可用 backend |

`flash_sim (py)` 在 Python 层实现分块逻辑，验证算法正确性，但不等于真正的 FlashAttention CUDA kernel（autograd 仍会保留中间变量）。`PyTorch SDPA (auto)` 在满足条件时会调用真实的 CUDA kernel，是实际使用中应选择的写法。

结果保存至 `results/03_benchmark_flash_attention.json`，供 Notebook 05 综合比较使用。

In [ ]:
import torch.utils.benchmark as benchmark


def measure_latency_memory(
    fn,
    *args,
    n_warmup: int = 5,
    n_repeat: int = 20,
    label: str = "",
) -> dict:
    """
    测量函数延迟（ms）和 GPU 峰值显存（MB）。
    CPU 模式下显存记为 NaN。
    """
    dev = args[0].device

    # 预热
    for _ in range(n_warmup):
        with torch.no_grad():
            fn(*args)

    # 显存峰值（仅 CUDA）
    if dev.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats(dev)
        with torch.no_grad():
            fn(*args)
        torch.cuda.synchronize()
        mem_mb = torch.cuda.max_memory_allocated(dev) / 1e6
    else:
        mem_mb = float("nan")

    # 延迟（使用 torch.utils.benchmark 消除方差）
    t = benchmark.Timer(
        stmt="fn(*args)",
        globals={"fn": fn, "args": args},
        label=label,
        num_threads=1,
    )
    result = t.timeit(n_repeat)

    return {
        "impl": label,
        "seq_len": args[0].shape[-2],
        "latency_ms": result.mean * 1e3,
        "latency_std_ms": result._median_relative_ci * result.mean * 1e3,
        "memory_mb": mem_mb,
        "dtype": str(args[0].dtype),
        "device": str(dev),
    }

In [ ]:
# ── 基准测试配置 ───────────────────────────────────────────────────────────────
BATCH, HEADS, D_HEAD = 1, 8, 64

# CPU 上只测小序列；GPU 上测更大范围
if device == "cuda":
    SEQ_LENS = [256, 512, 1024, 2048, 4096]
else:
    SEQ_LENS = [64, 128, 256, 512]

# ── 实现包装函数 ───────────────────────────────────────────────────────────────

def run_standard(Q, K, V):
    """Math backend（标准朴素注意力）"""
    if Q.device.type == "cuda":
        with torch.backends.cuda.sdp_kernel(
            enable_flash=False, enable_math=True, enable_mem_efficient=False
        ):
            return F.scaled_dot_product_attention(Q, K, V)
    return F.scaled_dot_product_attention(Q, K, V)


def run_flash_sim(Q, K, V):
    """Algorithm 1 PyTorch 模拟"""
    out, _, _ = flash_attention_v1_forward(Q, K, V, block_r=64, block_c=64)
    return out


def run_sdpa_default(Q, K, V):
    """PyTorch 默认 SDPA（GPU+fp16 时自动用 Flash Attention kernel）"""
    return F.scaled_dot_product_attention(Q, K, V)


implementations = [
    ("standard (math)", run_standard),
    ("flash_sim (py)", run_flash_sim),
    ("PyTorch SDPA (auto)", run_sdpa_default),
]

# ── 运行基准测试 ───────────────────────────────────────────────────────────────
all_records = []

print(f"设备：{device}  |  batch={BATCH}, heads={HEADS}, d={D_HEAD}")
print(f"{'N':>6}  {'实现':>20}  {'延迟(ms)':>10}  {'显存(MB)':>10}")
print("-"* 55)

for N in SEQ_LENS:
    torch.manual_seed(0)
    # fp16（GPU）或 fp32（CPU）
    dtype = torch.float16 if device == "cuda"else torch.float32
    Qb = torch.randn(BATCH, HEADS, N, D_HEAD, device=device, dtype=dtype)
    Kb = torch.randn(BATCH, HEADS, N, D_HEAD, device=device, dtype=dtype)
    Vb = torch.randn(BATCH, HEADS, N, D_HEAD, device=device, dtype=dtype)

    for name, fn in implementations:
        try:
            rec = measure_latency_memory(fn, Qb, Kb, Vb, label=name)
            all_records.append(rec)
            print(f"{N:>6}  {name:>20}  {rec['latency_ms']:>10.3f}  {rec['memory_mb']:>10.1f}")
        except Exception as e:
            print(f"{N:>6}  {name:>20}  SKIP: {e}")

# 保存 JSON 结果（供 Notebook 05 使用）
with open("../results/03_benchmark_flash_attention.json", "w") as f:
    json.dump(all_records, f, indent=2)
print(f"\n基准数据已保存至 results/03_benchmark_flash_attention.json")

In [ ]:
import pandas as pd

df_bench = pd.DataFrame(all_records)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {
    "standard (math)": "tomato",
    "flash_sim (py)": "orange",
    "PyTorch SDPA (auto)": "steelblue",
}
markers = {"standard (math)": "s", "flash_sim (py)": "D", "PyTorch SDPA (auto)": "o"}

for ax, metric, ylabel, title in [
    (axes[0], "latency_ms", "Latency (ms)", "Latency vs Sequence Length"),
    (axes[1], "memory_mb", "Peak Memory (MB)", "Peak Memory vs Sequence Length"),
]:
    for impl, grp in df_bench.groupby("impl"):
        g = grp.sort_values("seq_len")
        valid = g.dropna(subset=[metric])
        if valid.empty or valid[metric].isna().all():
            continue
        ax.plot(
            valid["seq_len"], valid[metric],
            marker=markers.get(impl, "o"),
            color=colors.get(impl),
            linewidth=1.8,
            label=impl,
        )
    ax.set_xlabel("Sequence Length  $N$", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

dtype_label = "fp16"if device == "cuda"else "fp32"
plt.suptitle(
    f"Flash Attention Benchmark  |  B={BATCH}, H={HEADS}, d={D_HEAD}, {dtype_label}, {device}",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
plt.savefig("../results/03_benchmark_plot.png", dpi=120, bbox_inches="tight")
plt.show()
print("图像已保存至 results/03_benchmark_plot.png")

In [ ]:
# ── 加速比与显存节省分析 ───────────────────────────────────────────────────────
pivot_lat = df_bench.pivot_table(index="seq_len", columns="impl", values="latency_ms")
pivot_mem = df_bench.pivot_table(index="seq_len", columns="impl", values="memory_mb")

std_col  = "standard (math)"
sdpa_col = "PyTorch SDPA (auto)"

if std_col in pivot_lat.columns and sdpa_col in pivot_lat.columns:
    pivot_lat["speedup (F.sdpa / standard)"] = (
        pivot_lat[std_col] / pivot_lat[sdpa_col]
    )
    print("=== 延迟加速比（standard / F.sdpa default）===")
    print(pivot_lat[[std_col, sdpa_col, "speedup (F.sdpa / standard)"]]
          .to_string(float_format="{:.3f}".format))

if std_col in pivot_mem.columns and sdpa_col in pivot_mem.columns:
    pivot_mem["mem_reduction (standard / F.sdpa)"] = (
        pivot_mem[std_col] / pivot_mem[sdpa_col]
    )
    print("\n=== 显存节省比（standard / F.sdpa default）===")
    print(pivot_mem[[std_col, sdpa_col, "mem_reduction (standard / F.sdpa)"]]
          .to_string(float_format="{:.3f}".format))
else:
    print("（CPU 模式无显存数据，跳过显存对比）")

In [ ]:
# ── 序列长度可扩展性：拟合增长斜率 ──────────────────────────────────────────────
from scipy import stats as sp_stats

print("=== 延迟随序列长度的增长斜率（log-log 线性回归）===")
print(f"{'实现':>22}  {'斜率':>8}  {'拟合 R²':>8}  {'理论斜率'}")
print("-"* 58)

theory = {
    "standard (math)": "≈2.0 (O(N²))",
    "flash_sim (py)": "≈2.0 (python loop O(N²))",
    "PyTorch SDPA (auto)": "≈2.0 (O(N²d²/M), 斜率相同但常数小)",
}

for impl, grp in df_bench.groupby("impl"):
    g = grp.sort_values("seq_len").dropna(subset=["latency_ms"])
    if len(g) < 3:
        continue
    log_n = np.log(g["seq_len"].values)
    log_t = np.log(g["latency_ms"].values)
    slope, _, r, _, _ = sp_stats.linregress(log_n, log_t)
    r2 = r**2
    theory_str = theory.get(impl, "")
    print(f"{impl:>22}  {slope:>8.2f}  {r2:>8.4f}  {theory_str}")

## 5 反向传播：重计算策略

### 5.1 标准实现的反向传播问题

标准注意力的反向传播需要 $A \in \mathbb{R}^{N \times N}$（注意力权重矩阵）来计算梯度：
$$
\frac{\partial L}{\partial V} = A^\top \frac{\partial L}{\partial O}, \quad
\frac{\partial L}{\partial Q} \propto A \cdot (\ldots), \quad
\frac{\partial L}{\partial K} \propto A^\top \cdot (\ldots)
$$

因此需要将 $A$ 保存在显存中直到反向传播，这是 $O(N^2)$ 的额外显存消耗。

### 5.2 重计算（Recomputation）策略

FlashAttention 的做法：**前向传播不保存 $S$ 或 $A$**，只保存较小的统计量，反向传播时**重新计算**所需的中间值。

**前向传播保存**（$O(Nd)$ 量级）：
- 输出矩阵 $O \in \mathbb{R}^{N \times d}$
- 对数归一化因子：$\text{logsumexp}^{(i)} = m^{(i)} + \ln \ell^{(i)}$，每行一个标量，共 $N$ 个
- 输入 $Q, K, V$（前向传播的中间激活，反向传播计算梯度时本就需要保留；$W_Q, W_K, W_V$ 才是模型参数）

**反向传播流程**（也分块进行）：
```
for j = 1 to Tc:                        # 外循环 K/V 块
    从 HBM 加载 Kⱼ, Vⱼ
    for i = 1 to Tr:                     # 内循环 Q 块
        从 HBM 加载 Qᵢ, Oᵢ, dOᵢ, logsumexp_i
        重新计算 Sᵢⱼ = Qᵢ·Kⱼᵀ/√d         （SRAM 内）
        重新计算 Aᵢⱼ = softmax(Sᵢⱼ)       （SRAM 内）
        计算 dVⱼ += Aᵢⱼᵀ · dOᵢ
        计算 dAᵢⱼ = dOᵢ · Vⱼᵀ
        计算 dSᵢⱼ = Aᵢⱼ ⊙ (dAᵢⱼ − rowsum(dAᵢⱼ ⊙ Aᵢⱼ))
        计算 dQᵢ += dSᵢⱼ · Kⱼ
        计算 dKⱼ += dSᵢⱼᵀ · Qᵢ
```

### 5.3 显存对比

| | 标准反向 | FlashAttention 反向 |
|--|----------|---------------------|
| 保存的中间量 | $S, A \in O(N^2)$ | $O, \text{lse} \in O(Nd)$ |
| 反向重算 | 无需重算 | 重算 $S_{ij}, A_{ij}$（SRAM 内） |
| **总额外显存** | **$O(N^2)$** | **$O(Nd)$（即 $O(N)$，$d$ 为常数）** |
| HBM 访问代价 | 一次 | 两次（前向+反向各一次） |

### 5.4 `log-sum-exp` 技巧

实践中，FlashAttention 保存的是 $\text{lse}_i = m_i + \ln \ell_i$（log-sum-exp 值），而非分开保存 $m_i$ 和 $\ell_i$。这样反向传播只需一次减法和一次 `exp` 即可还原 $A_{ij}$ 的每行：
$$
A_{ij,k} = \exp(S_{ij,k} - \text{lse}_i)
$$

In [ ]:
# ── 演示 log-sum-exp 保存与反向重计算 ─────────────────────────────────────────
torch.manual_seed(5)
B_g, H_g, N_g, d_g = 1, 2, 32, 16

Qg = torch.randn(B_g, H_g, N_g, d_g, requires_grad=True, device=device, dtype=torch.float32)
Kg = torch.randn(B_g, H_g, N_g, d_g, requires_grad=True, device=device, dtype=torch.float32)
Vg = torch.randn(B_g, H_g, N_g, d_g, requires_grad=True, device=device, dtype=torch.float32)

# ── 前向：flash_attention_v1_forward 返回 (O, l, m)
Qg_fa = Qg.detach().clone().requires_grad_(True)
Kg_fa = Kg.detach().clone().requires_grad_(True)
Vg_fa = Vg.detach().clone().requires_grad_(True)

O_fa, l_fa, m_fa = flash_attention_v1_forward(Qg_fa, Kg_fa, Vg_fa, block_r=16, block_c=16)

# 计算 log-sum-exp（前向保存的统计量）
lse = m_fa + torch.log(l_fa)   # (B,H,N)  每行一个标量
print(f"前向保存的 lse 形状: {lse.shape}")
print(f"lse 总元素数: {lse.numel()}  vs  A 矩阵元素数: {B_g*H_g*N_g*N_g}")
print(f"→ 显存节省：{B_g*H_g*N_g*N_g / lse.numel():.0f}×（序列 N={N_g} 时）")

# ── 用 lse 重还原注意力权重 A 的某一行（反向传播中的重计算）
# 以第 0 batch, 0 head, 第 5 个 query 为例
i_q = 5
scale_g = 1.0 / math.sqrt(d_g)
with torch.no_grad():
    s_row = (Qg[0, 0, i_q] @ Kg[0, 0].T) * scale_g   # (N,)
    A_recomputed = torch.exp(s_row - lse[0, 0, i_q])  # 用 lse 直接还原
    A_reference  = torch.softmax(s_row, dim=0)         # 标准 softmax
    err_lse = (A_recomputed - A_reference).abs().max().item()

print(f"\n反向重计算 A 第 {i_q} 行 vs 标准 softmax 最大误差: {err_lse:.2e}")
assert err_lse < 1e-5
print("✓ lse 重计算正确，精度与直接 softmax 相当")

# ── 验证 autograd 可以通过 flash_attention_v1_forward 反向传播
loss = O_fa.sum()
loss.backward()
print(f"\n梯度验证（dQ/dK/dV 均非 None）：")
print(f"dQ: {Qg_fa.grad.shape}, max={Qg_fa.grad.abs().max():.4f}")
print(f"dK: {Kg_fa.grad.shape}, max={Kg_fa.grad.abs().max():.4f}")
print(f"dV: {Vg_fa.grad.shape}, max={Vg_fa.grad.abs().max():.4f}")

In [ ]:
# ── 显存随序列长度的增长：O(N²) vs O(N) 可视化 ───────────────────────────────
import matplotlib.patches as mpatches

d_mem = 64
seq_mems = [128, 256, 512, 1024, 2048, 4096, 8192]

# 标准注意力反向：保存 A(N²) + S(N²) 个 fp32 元素
standard_mem_mb = [2 * N**2 * 4 / 1e6 for N in seq_mems]
# FlashAttention 反向：保存 O(N×d) + lse(N) 个 fp32 元素
flash_mem_mb    = [(N * d_mem + N) * 4 / 1e6 for N in seq_mems]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(seq_mems, standard_mem_mb, marker="s", color="tomato",
            linewidth=2, label=r"Standard: $O(N^2)$  (saves $S$, $A$)")
ax.semilogy(seq_mems, flash_mem_mb, marker="o", color="steelblue",
            linewidth=2, label=r"FlashAttention: $O(N)$  (saves $O$, lse)")

ax.set_xlabel("Sequence Length  $N$", fontsize=11)
ax.set_ylabel("Backward Extra Memory  (MB, log scale)", fontsize=11)
ax.set_title(f"Memory Complexity of Backward Pass  ($d$={d_mem})")
ax.set_xticks(seq_mems)
ax.set_xticklabels([str(n) for n in seq_mems], rotation=30)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("../results/03_backward_memory.png", dpi=120, bbox_inches="tight")
plt.show()
print("图像已保存至 results/03_backward_memory.png")

## 6 FlashAttention v2 的主要改进

FlashAttention v2（Dao, ICLR 2024）在 v1 的 IO 感知 Tiling 基础上，针对现代 GPU（A100/H100）的计算特性做了进一步优化，主要集中在三个方向：减少 HBM 写操作、减少 non-matmul FLOPs、提升 GPU occupancy。

### 6.1 减少 $O_i$ 的 HBM 写操作

v1 的循环结构是**外层遍历 $K/V$ 块、内层遍历 $Q$ 块**。每次内层迭代结束时都要把 $O_i$、$l_i$、$m_i$ 写回 HBM，共写 $T_c \times T_r$ 次。

v2 把循环顺序调换为**外层遍历 $Q$ 块、内层遍历 $K/V$ 块**。这样 $O_i$、$l_i$、$m_i$ 可以在整个内层循环（遍历完所有 $K/V$ 块）期间一直保留在 SRAM 中，只在外层每次迭代结束时写回 HBM 一次，共写 $T_r$ 次。

```
v1 写 O 次数：T_c × T_r 次
v2 写 O 次数：T_r 次（节省 T_c 倍）
```

这对 $T_c$ 较大（长序列、小 tile）的场景效果显著，同时也有助于减少 warp 之间的同步开销。

### 6.2 减少 non-matmul FLOPs 与输出累积优化

v1 中每步内层迭代均需归一化输出：
$$
O_i^{\text{new}} = \text{diag}(l_i^{\text{new}})^{-1} \!\left(
    \text{diag}(l_i)\, e^{m_i - m_i^{\text{new}}} O_i
    + e^{\tilde{m}_{ij} - m_i^{\text{new}}} \tilde{P}_{ij} V_j
\right)
$$

v2 改为维护**未归一化**的 $\tilde{O}_i$，仅在外层迭代结束时做一次归一化：
$$
\tilde{O}_i^{\text{new}} = e^{m_i - m_i^{\text{new}}} \tilde{O}_i + e^{\tilde{m}_{ij} - m_i^{\text{new}}} \tilde{P}_{ij} V_j
$$

这省去了每步的 $\text{diag}(l_i^{\text{new}})^{-1}$ 逐行除法（non-matmul FLOPs），对 $T_c$ 步内层迭代共节省 $T_c - 1$ 次除法。$QK^\top$ 和 $PV$ 的矩阵乘 FLOPs 不变。

### 6.3 因果掩码优化

v2 外层遍历 $Q$ 块时，可以提前判断当前 $K/V$ 块与当前 $Q$ 块的因果遮蔽关系：

- 若 $K$ 块的所有位置均在 $Q$ 块的最小位置之后（全部遮蔽）：**跳过该 $K/V$ 块**，无需加载或计算
- 若 $K$ 块的所有位置均在 $Q$ 块的最大位置之前（全部可见）：**正常计算，无需逐元素掩码**
- 仅对跨越对角线的块才真正应用掩码（约 $T_r$ 个块）

跳过的 $K/V$ 块约占总块数的一半（上三角），使因果注意力的实际 FLOPs 接近全注意力的 50%。实测延迟约为全注意力的 60–70%（受 kernel 启动开销和边界块处理影响）。

### 6.4 Thread-block Work Partitioning 与 GPU Occupancy

v2 另一个重要改进是更好地利用 GPU 的并行资源。

在 v1 中，一个 thread block 负责一个 $(Q_i, \text{所有} K/V)$ 或 $(\text{所有} Q, K_j/V_j)$ 的计算，不同 thread block 之间的工作负载可能不均衡，且对于短序列，活跃 thread block 数量少，GPU occupancy 低。

v2 沿 sequence dimension 将工作更细粒度地分配给更多 thread block，使 GPU 的 SM（Streaming Multiprocessor）保持更高的占用率。**这里的"序列并行"指的是单 GPU 内让更多 thread block 沿序列维度并行工作**，而不是把序列切分到多张 GPU。

> 多 GPU 的序列并行（如 Ring Attention）是独立的系统级技术，利用 $(m, \ell)$ 的结合律跨设备合并统计量，不属于 FlashAttention v2 本身的改进。

In [ ]:
# ── FlashAttention v2：外循环换成 Q 块，维护未归一化输出 ─────────────────────
def flash_attention_v2_forward(
    Q: torch.Tensor, # (B, H, N, d)
    K: torch.Tensor, # (B, H, N, d)
    V: torch.Tensor, # (B, H, N, d)
    block_r: int = 64,
    block_c: int = 64,
    causal: bool = False,
) -> torch.Tensor:
    """
    FlashAttention v2 前向：外循环遍历 Q 块，维护未归一化输出 Õᵢ。

    v1 vs v2 的关键区别：
    - 外/内循环互换（Q 块在外）
    - Oᵢ 在整个内循环中驻留 SRAM，仅在内循环结束时写回 HBM 一次
    - 维护未归一化 Õᵢ，最后一次性归一化（减少每步除法）
    - 因果注意力跳过全遮蔽块
    """
    B, H, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    Tr = math.ceil(N / block_r)
    Tc = math.ceil(N / block_c)

    O = torch.zeros_like(Q)

    # v2 外循环：Q 块（每个 Q 块独立，天然支持序列并行）
    for i in range(Tr):
        i0, i1 = i * block_r, min((i + 1) * block_r, N)
        Qi = Q[:,:, i0:i1,:]          # (B,H,Br',d)  从 HBM 加载

        # SRAM 中维护该 Q 块的累积量（不在每步写回 HBM）
        mi = torch.full((B, H, i1 - i0), float("-inf"), device=Q.device, dtype=Q.dtype)
        li = torch.zeros(B, H, i1 - i0, device=Q.device, dtype=Q.dtype)
        Oi_unnorm = torch.zeros(B, H, i1 - i0, d, device=Q.device, dtype=Q.dtype)  # 未归一化

        # v2 内循环：K/V 块
        for j in range(Tc):
            j0, j1 = j * block_c, min((j + 1) * block_c, N)

            # 因果优化：跳过全遮蔽块（j0 > i1-1 意味着所有 key 位置 > 最大 query 位置）
            if causal and j0 > i1 - 1:
                break  # 后续 j 块更大，全部遮蔽，直接退出内循环

            Kj = K[:,:, j0:j1,:]      # (B,H,Bc',d)  从 HBM 加载
            Vj = V[:,:, j0:j1,:]      # (B,H,Bc',d)

            Sij = torch.matmul(Qi, Kj.transpose(-2, -1)) * scale  # (B,H,Br',Bc')

            # 因果掩码（仅对对角线附近的块生效）
            if causal:
                q_pos = torch.arange(i0, i1, device=Q.device).view(1, 1, -1, 1)
                k_pos = torch.arange(j0, j1, device=Q.device).view(1, 1, 1, -1)
                Sij = Sij.masked_fill(k_pos > q_pos, float("-inf"))

            m_tilde = Sij.amax(dim=-1)                              # (B,H,Br')
            P_tilde = torch.exp(Sij - m_tilde.unsqueeze(-1))       # (B,H,Br',Bc')
            l_tilde = P_tilde.sum(dim=-1)                           # (B,H,Br')

            # v2 更新：维护未归一化输出 Õᵢ（减少每步除法）
            mi_new = torch.maximum(mi, m_tilde)
            li_new = torch.exp(mi - mi_new) * li + torch.exp(m_tilde - mi_new) * l_tilde

            # Õᵢ 的更新不需要除以 l_new，每步少一个 N×d 的除法
            Oi_unnorm = (
                torch.exp(mi - mi_new).unsqueeze(-1) * Oi_unnorm
                + torch.exp(m_tilde - mi_new).unsqueeze(-1) * torch.matmul(P_tilde, Vj)
            )
            mi, li = mi_new, li_new

        # 内循环结束后，Q 块只写回 HBM 一次（v2 的关键优化）
        O[:,:, i0:i1,:] = Oi_unnorm / li.unsqueeze(-1)  # 最终归一化

    return O


# ── v1 vs v2 正确性验证 ────────────────────────────────────────────────────────
torch.manual_seed(0)
B_v, H_v, N_v, d_v = 2, 4, 128, 64
Qv2 = torch.randn(B_v, H_v, N_v, d_v, device=device)
Kv2 = torch.randn(B_v, H_v, N_v, d_v, device=device)
Vv2 = torch.randn(B_v, H_v, N_v, d_v, device=device)

if device == "cuda":
    with torch.backends.cuda.sdp_kernel(
        enable_flash=False, enable_math=True, enable_mem_efficient=False
    ):
        ref_v2 = F.scaled_dot_product_attention(Qv2, Kv2, Vv2)
        ref_v2c = F.scaled_dot_product_attention(Qv2, Kv2, Vv2, is_causal=True)
else:
    ref_v2 = F.scaled_dot_product_attention(Qv2, Kv2, Vv2)
    ref_v2c = F.scaled_dot_product_attention(Qv2, Kv2, Vv2, is_causal=True)

print("=== v2 正确性验证（非因果）===")
for bs in [16, 32, 64]:
    out_v2 = flash_attention_v2_forward(Qv2, Kv2, Vv2, block_r=bs, block_c=bs)
    err_v2 = (out_v2 - ref_v2).abs().max().item()
    print(f"块大小={bs:3d}: 误差={err_v2:.2e}  {'✓' if err_v2 < 1e-4 else '✗'}")

print("\n=== v2 正确性验证（因果，跳过全遮蔽块）===")
for bs in [16, 32, 64]:
    out_v2c = flash_attention_v2_forward(Qv2, Kv2, Vv2, block_r=bs, block_c=bs, causal=True)
    err_v2c = (out_v2c - ref_v2c).abs().max().item()
    print(f"块大小={bs:3d}: 误差={err_v2c:.2e}  {'✓' if err_v2c < 1e-4 else '✗'}")

In [ ]:
# ── v1 vs v2 延迟对比（演示循环顺序与 HBM 写次数的影响）────────────────────────
import torch.utils.benchmark as benchmark

def bench_fn(fn, *args, n=15):
    for _ in range(3): fn(*args)   # warmup
    t = benchmark.Timer(stmt="fn(*args)", globals={"fn": fn, "args": args}, num_threads=1)
    return t.timeit(n).mean * 1e3  # ms


print("=== v1 vs v2 延迟对比（PyTorch 模拟，展示算法差异）===")
print(f"{'N':>6}  {'v1 (ms)':>10}  {'v2 (ms)':>10}  {'v2/v1':>8}")
print("-"* 40)

bench_seq = [64, 128, 256] if device == "cpu"else [256, 512, 1024]
for N_b in bench_seq:
    torch.manual_seed(0)
    Qb = torch.randn(1, 4, N_b, 64, device=device)
    Kb = torch.randn(1, 4, N_b, 64, device=device)
    Vb = torch.randn(1, 4, N_b, 64, device=device)

    t_v1 = bench_fn(lambda q,k,v: flash_attention_v1_forward(q, k, v, block_r=64, block_c=64), Qb, Kb, Vb)
    t_v2 = bench_fn(lambda q,k,v: flash_attention_v2_forward(q, k, v, block_r=64, block_c=64), Qb, Kb, Vb)
    ratio = t_v2 / t_v1
    print(f"{N_b:>6}  {t_v1:>10.3f}  {t_v2:>10.3f}  {ratio:>8.3f}")

print()
print("注：此处 v1/v2 均为 Python 级别模拟，差异来自循环结构和输出写回次数。")
print("真正的 CUDA kernel 实现中 v2 比 v1 快约 2–3× (Dao et al., 2024)。")

In [ ]:
# ── 可视化 v2 因果优化：跳过的块比例 ────────────────────────────────────────────
block_sizes_vis = [32, 64, 128]
seq_lens_vis    = [512, 1024, 2048, 4096]

fig, ax = plt.subplots(figsize=(8, 5))

for bs in block_sizes_vis:
    skipped_fracs = []
    for N_vis in seq_lens_vis:
        Tr = math.ceil(N_vis / bs)
        Tc = math.ceil(N_vis / bs)
        total_blocks = Tr * Tc
        skipped = 0
        for i in range(Tr):
            i1 = min((i + 1) * bs, N_vis)
            for j in range(Tc):
                j0 = j * bs
                if j0 > i1 - 1:   # v2 的跳过条件
                    skipped += Tc - j  # 后续所有块都跳过
                    break
        skipped_fracs.append(skipped / total_blocks * 100)
    ax.plot(seq_lens_vis, skipped_fracs, marker="o", label=f"block_size={bs}")

ax.axhline(50, linestyle="--", color="gray", alpha=0.7, label="理论上界 50%")
ax.set_xlabel("Sequence Length  $N$", fontsize=11)
ax.set_ylabel("Skipped Blocks  (%)", fontsize=11)
ax.set_title("v2 Causal Optimization: Fraction of Skipped Blocks")
ax.legend(fontsize=9)
ax.set_ylim(0, 60)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../results/03_causal_skip.png", dpi=120, bbox_inches="tight")
plt.show()
print("图像已保存至 results/03_causal_skip.png")

## 7 小结

### 7.1 核心结论

| 维度 | 结论 |
|------|------|
| **精度等价性** | FlashAttention 与标准注意力数学完全等价（精确算法）；不同 backend 的误差仅来自浮点运算顺序，FP16 下通常 $\leq 10^{-3}$ |
| **IO 复杂度** | HBM 访问量从 $\Theta(N^2)$（主导项）降至 $\Theta(N^2 d^2/M)$，理论节省 $M/d^2$ 倍 |
| **显存** | 不 materialize $N\times N$ 的 $S$/$A$；前向+反向合计额外显存 $O(Nd)$，相比标准的 $O(N^2)$ 节省显著 |
| **延迟** | GPU + fp16 实测加速 2–4×；HBM 带宽是瓶颈（而非算力）时效果最明显 |
| **v2 主要改进** | 外层遍历 $Q$ 块减少 $O_i$ 写回次数；省去每步 non-matmul 除法；因果跳块约减半 FLOPs；更好的 thread-block 工作划分提升 GPU occupancy |
| **PyTorch 接口** | `F.scaled_dot_product_attention` 默认自动选择 backend；需要确认 backend 或做性能对比时，用 `sdp_kernel` 显式控制 |

### 7.2 适用场景

| 场景 | 推荐 |
|------|------|
| 长序列（$N \gtrsim 1024$）训练/推理 | ✅ 显存节省最显著，带宽收益大 |
| CUDA 设备 + fp16/bf16 + head_dim ≤ 128 | ✅ PyTorch SDPA auto 模式通常会选到 Flash backend |
| CPU 推理 | ⚠️ 无 Flash/Efficient backend，降回 math |
| 需要任意形状的自定义掩码 | ⚠️ 可能无法触发 Flash backend，考虑 mem-efficient 或 math |
| 极小序列（$N < 64$）+ 小 batch | ⚠️ kernel 启动开销可能大于收益 |

### 7.3 下一步：量化

FlashAttention 解决了长序列场景下 HBM 带宽瓶颈，$Q$、$K$、$V$ 本身仍以 FP16/FP32 存储。另一个正交的优化方向是**量化**：将激活值和权重压缩到更低比特宽度，在精度可接受的前提下进一步减少显存与计算量。

`04_quantization.ipynb` 将聚焦于：
- 注意力层中 $Q$、$K$、$V$ 及注意力权重的量化挑战
- 逐张量 vs 逐通道量化对精度的影响
- `torch.quantization` / `bitsandbytes` 工具链实操
- 量化前后延迟和显存与本节 FlashAttention 基准的横向对比

---
*参考文献*
- Dao et al. "FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness", NeurIPS 2022.
- Dao. "FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning", ICLR 2024.
- Milakov & Gimelshein. "Online normalizer calculation for softmax", arXiv:1805.02867, 2018.
- PyTorch Docs: [scaled_dot_product_attention](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)